In [ ]:
import pandas as pd
from pathlib import Path
from collections import defaultdict

# ================================================
# CONFIG
# ================================================
FILE_PATH = Path("Copy of Master Data_290102026 2 - Copy (wecompress.com).xlsx")
MASTER_SHEET = "Master Data "
CATEGORY_SHEET = "Sheet1"               # PARTNO, Category

MAX_HOURS_PER_DAY    = 22.0
CHANGEOVER_HOURS     = 40 / 60.0
MAX_PARTS_PER_MACHINE = 3

# ================================================
# MACHINE NAME NORMALIZATION
# ================================================
def normalize_machine(s):
    if pd.isna(s) or not str(s).strip():
        return None
    s = str(s).strip().upper()
    # Aggressive normalization
    s = s.replace(".", "-").replace("M.P-", "MP-").replace("MP.", "MP-")
    s = s.replace("TOYOI", "TOYO").replace("TOYO ", "TOYO-").replace("TOYOIST", "TOYO-IST")
    s = s.replace("TOYO-6TH", "TOYO-6TH")      # keep as is
    s = s.replace(" ", "-").replace("_", "-")
    if len(s) < 3 or "N/A" in s or "#" in s:
        return None
    return s

def get_machines(cell):
    if pd.isna(cell):
        return []
    parts = str(cell).split(",")
    cleaned = [normalize_machine(x) for x in parts if normalize_machine(x)]
    return list(set(cleaned))

# ================================================
# MAIN LOGIC
# ================================================

print("Step 1: Loading category mapping...")
cat_df = pd.read_excel(FILE_PATH, sheet_name=CATEGORY_SHEET)
cat_df = cat_df[["PARTNO", "Category"]].dropna(subset=["PARTNO"])
cat_map = dict(zip(cat_df["PARTNO"], cat_df["Category"]))
print(f"→ {len(cat_map):,} part-category mappings loaded")

print("\nStep 2: Loading & aggregating Master Data...")
master = pd.read_excel(FILE_PATH, sheet_name=MASTER_SHEET)

for col in ["Daily Plan", "Sub Count", "Inventory_25", "Minimum Quantity", "Cycle Time"]:
    if col in master.columns:
        master[col] = pd.to_numeric(master[col], errors="coerce").fillna(0)

records = []
all_machines_seen = set()

for child, g in master.groupby("Child Part"):
    daily_demand = (g["Daily Plan"] * g["Sub Count"]).sum()
    if daily_demand <= 0:
        continue

    net_req = daily_demand + g["Minimum Quantity"].iloc[0] - g["Inventory_25"].iloc[0]
    if net_req <= 0:
        continue

    cycle_valid = g["Cycle Time"][g["Cycle Time"] > 0]
    if cycle_valid.empty:
        continue
    cycle_sec = cycle_valid.iloc[0]

    vm_raw = g["Vertical Machines"].dropna().unique()
    if len(vm_raw) == 0:
        continue

    machines = get_machines(",".join(vm_raw.astype(str)))
    if not machines:
        continue

    all_machines_seen.update(machines)

    records.append({
        "Child Part": child,
        "Daily_Demand": daily_demand,
        "Net_Required": net_req,
        "Cycle_Time_sec": cycle_sec,
        "Eligible_Machines": machines,
        "Category": cat_map.get(child, "Unknown")
    })

df = pd.DataFrame(records)
print(f"\n→ {len(df):,} parts kept after filtering")

print("\nCategories used (from Sheet1):")
print(df["Category"].value_counts(dropna=False))

print("\nAll unique machine names found in data:")
print(sorted(all_machines_seen))

# Only plan Repeaters + Strangers
to_schedule = df[df["Category"].isin(["Repeater", "Stranger"])].copy()
print(f"\nParts to schedule: {len(to_schedule):,}")

if len(to_schedule) == 0:
    print("No parts to schedule → check if categories are correctly loaded")
    exit()

# ================================================
# SCHEDULER – now safe against unknown machines
# ================================================

machine_load = defaultdict(float)       # auto-creates entries
machine_sequence = defaultdict(list)
schedule = []

print("\nRunning scheduler...")
to_schedule = to_schedule.sort_values("Net_Required", ascending=False)

for _, row in to_schedule.iterrows():
    hrs_per_pc = row["Cycle_Time_sec"] / 3600.0
    if hrs_per_pc <= 0:
        continue

    qty_left = row["Net_Required"]
    eligible = row["Eligible_Machines"]

    for m in sorted(eligible, key=lambda x: machine_load[x]):
        if len(machine_sequence[m]) >= MAX_PARTS_PER_MACHINE:
            continue

        free = MAX_HOURS_PER_DAY - machine_load[m]
        if free <= CHANGEOVER_HOURS:
            continue

        setup_h = CHANGEOVER_HOURS
        free_after = free - setup_h
        if free_after <= 0:
            continue

        max_qty = free_after / hrs_per_pc
        assign_qty = min(qty_left, max_qty)

        if assign_qty < 10:
            continue

        assign_h = assign_qty * hrs_per_pc

        machine_load[m] += assign_h + setup_h
        machine_sequence[m].append({
            "Child Part": row["Child Part"],
            "Qty": round(assign_qty),
            "Hours": round(assign_h + setup_h, 2),
            "Category": row["Category"]
        })

        schedule.append({
            "Machine": m,
            "Child Part": row["Child Part"],
            "Qty": round(assign_qty),
            "Hours": round(assign_h + setup_h, 2),
            "Category": row["Category"]
        })

        qty_left -= assign_qty
        if qty_left <= 0:
            break

# ================================================
# OUTPUT
# ================================================

print("\n" + "="*90)
print("FINAL PLAN – Repeaters & Strangers only")
print("="*90)

total_h = sum(machine_load.values())
for m, h in sorted(machine_load.items(), key=lambda x: x[1], reverse=True):
    seq = machine_sequence[m]
    if not seq:
        continue
    print(f"\n{m:12}  {h:6.1f}h  ({h/MAX_HOURS_PER_DAY*100:5.1f}%)  {len(seq)} parts")
    for p in seq:
        print(f"  • {p['Child Part']:20}  {p['Qty']:>6} pcs   {p['Hours']:>5.1f}h  {p['Category']}")

print("\n" + "-"*90)
print(f"Total hours: {total_h:.1f}")
print(f"Parts planned: {len(schedule)}")
print("-"*90)

if schedule:
    pd.DataFrame(schedule).to_excel("production_plan_final.xlsx", index=False)
    print("Saved → production_plan_final.xlsx")

In [ ]:
import pandas as pd
from pathlib import Path
from collections import defaultdict

# ================================================
# CONFIG – ONLY these 4 machines
# ================================================
FILE_PATH = Path("Copy of Master Data_290102026 2 - Copy (wecompress.com).xlsx")
MASTER_SHEET   = "Master Data "
CATEGORY_SHEET = "Sheet1"

ALLOWED_MACHINES = ["MP-01", "MP-05", "MP-10", "MP-17"]

MAX_HOURS_PER_DAY    = 22.0
CHANGEOVER_HOURS     = 40 / 60.0
MAX_PARTS_PER_MACHINE = 3

# ================================================
# MACHINE NORMALIZATION – strict filter to only 4 machines
# ================================================
def normalize_machine(s):
    if pd.isna(s) or not str(s).strip():
        return None
    s = str(s).strip().upper()
    s = s.replace(".", "-").replace("M.P-", "MP-").replace("MP.", "MP-")
    s = s.replace(" ", "-")
    return s

def get_allowed_machines(cell):
    if pd.isna(cell):
        return []
    parts = str(cell).split(",")
    cleaned = [normalize_machine(x) for x in parts if normalize_machine(x)]
    return [m for m in set(cleaned) if m in ALLOWED_MACHINES]

# ================================================
# LOAD CATEGORY MAPPING
# ================================================
cat_df = pd.read_excel(FILE_PATH, sheet_name=CATEGORY_SHEET)
cat_df = cat_df[["PARTNO", "Category"]].dropna(subset=["PARTNO"])
cat_map = dict(zip(cat_df["PARTNO"], cat_df["Category"]))

# ================================================
# AGGREGATE MASTER DATA
# ================================================
master = pd.read_excel(FILE_PATH, sheet_name=MASTER_SHEET)

for col in ["Daily Plan", "Sub Count", "Inventory_25", "Minimum Quantity", "Cycle Time"]:
    if col in master.columns:
        master[col] = pd.to_numeric(master[col], errors="coerce").fillna(0)

records = []
for child, g in master.groupby("Child Part"):
    daily_demand = (g["Daily Plan"] * g["Sub Count"]).sum()
    if daily_demand <= 0:
        continue

    net_req = daily_demand + g["Minimum Quantity"].iloc[0] - g["Inventory_25"].iloc[0]
    if net_req <= 0:
        continue

    cycle_valid = g["Cycle Time"][g["Cycle Time"] > 0]
    if cycle_valid.empty:
        continue
    cycle_sec = cycle_valid.iloc[0]

    machines = get_allowed_machines(",".join(g["Vertical Machines"].dropna().astype(str)))
    if not machines:
        continue

    records.append({
        "Child Part": child,
        "Daily_Demand": daily_demand,
        "Net_Required": net_req,
        "Cycle_Time_sec": cycle_sec,
        "Eligible_Machines": machines,
        "Category": cat_map.get(child, "Unknown")
    })

df = pd.DataFrame(records)

# Ensure 'Category' column always exists
if "Category" not in df.columns:
    df["Category"] = "Unknown"

# Only Repeater + Stranger
to_schedule = df[df["Category"].isin(["Repeater", "Stranger"])].copy()

# ================================================
# SCHEDULER – ONLY on the 4 machines
# ================================================
machine_load = {m: 0.0 for m in ALLOWED_MACHINES}
machine_sequence = {m: [] for m in ALLOWED_MACHINES}
schedule = []

to_schedule = to_schedule.sort_values("Net_Required", ascending=False)

for _, row in to_schedule.iterrows():
    hrs_per_pc = row["Cycle_Time_sec"] / 3600.0
    if hrs_per_pc <= 0:
        continue

    qty_left = row["Net_Required"]
    eligible = row["Eligible_Machines"]  # already filtered to only the 4 machines

    for m in sorted(eligible, key=lambda x: machine_load[x]):
        if len(machine_sequence[m]) >= MAX_PARTS_PER_MACHINE:
            continue

        free = MAX_HOURS_PER_DAY - machine_load[m]
        if free <= CHANGEOVER_HOURS:
            continue

        setup_h = CHANGEOVER_HOURS
        free_after = free - setup_h
        if free_after <= 0:
            continue

        max_qty = free_after / hrs_per_pc
        assign_qty = min(qty_left, max_qty)
        if assign_qty < 10:
            continue

        assign_h = assign_qty * hrs_per_pc

        machine_load[m] += assign_h + setup_h
        machine_sequence[m].append({
            "Child Part": row["Child Part"],
            "Qty": round(assign_qty),
            "Hours": round(assign_h + setup_h, 2),
            "Category": row["Category"]
        })

        schedule.append({
            "Machine": m,
            "Child Part": row["Child Part"],
            "Qty": round(assign_qty),
            "Hours": round(assign_h + setup_h, 2),
            "Category": row["Category"]
        })

        qty_left -= assign_qty
        if qty_left <= 0:
            break

# ================================================
# DISPLAY PLAN IN CONSOLE ONLY
# ================================================
print("\n" + "="*90)
print("FINAL PLAN – ONLY MP-01, MP-05, MP-10, MP-17")
print("="*90)

total_hours = sum(machine_load.values())

for m in ALLOWED_MACHINES:
    seq = machine_sequence.get(m, [])
    h = machine_load.get(m, 0.0)
    if h < 0.1 and not seq:
        print(f"\n{m} → No parts assigned")
        continue

    print(f"\n🛠 {m}   {h:6.1f} / 22.0 h   ({h/22*100:5.1f}%)   {len(seq)} parts")
    for p in seq:
        print(f"   • {p['Child Part']:22}   {p['Qty']:>7,} pcs   {p['Hours']:>5.1f}h   {p['Category']}")

print("\n" + "-"*90)
print(f"Total hours used : {total_hours:.1f} h")
print(f"Parts scheduled  : {len(schedule)}")
print("-"*90)

In [ ]:
print(df[df["Category"].isin(["Repeater", "Stranger"])][["Child Part", "Category", "Eligible_Machines"]].head(15))